In [1]:
import polars as pl
from youtube_transcript_api import YouTubeTranscriptApi

In [2]:
def extract_text(transcript: list) -> str:
    """
        Function to extract text from transcript dictionary
    """
    
    text_list = [transcript[i]['text'] for i in range(len(transcript))]
    return ' '.join(text_list)

In [3]:
df = pl.read_parquet('video-ids.parquet')
print(df.head())

shape: (5, 3)
┌─────────────┬──────────────────────┬───────────────────────────────────┐
│ video_id    ┆ datetime             ┆ title                             │
│ ---         ┆ ---                  ┆ ---                               │
│ str         ┆ str                  ┆ str                               │
╞═════════════╪══════════════════════╪═══════════════════════════════════╡
│ _1f-o0nqpEI ┆ 2025-02-03T00:12:13Z ┆ DeepSeek, China, OpenAI, NVIDIA,… │
│ OHWnPOKh_S0 ┆ 2025-01-26T20:47:42Z ┆ Marc Andreessen: Trump, Power, T… │
│ Rz-4ulRKnz4 ┆ 2025-01-19T19:27:52Z ┆ Jennifer Burns: Milton Friedman,… │
│ u321m25rKXc ┆ 2025-01-05T19:00:47Z ┆ Volodymyr Zelenskyy: Ukraine, Wa… │
│ yhZAXXI83-4 ┆ 2024-12-22T22:31:58Z ┆ Adam Frank: Alien Civilizations … │
└─────────────┴──────────────────────┴───────────────────────────────────┘


In [4]:
%%time
transcript_text_list = []

for i in range(len(df)):

    # try to extract captions
    try:
        transcript = YouTubeTranscriptApi.get_transcript(df['video_id'][i])
        transcript_text = extract_text(transcript)
    # if not available set as n/a
    except:
        transcript_text = "n/a"
    
    transcript_text_list.append(transcript_text)

CPU times: total: 20.9 s
Wall time: 8min 28s


In [5]:
df = df.with_columns(pl.Series(name="transcript", values=transcript_text_list))
print(df.head())

shape: (5, 4)
┌─────────────┬──────────────────────┬──────────────────────────┬──────────────────────────────────┐
│ video_id    ┆ datetime             ┆ title                    ┆ transcript                       │
│ ---         ┆ ---                  ┆ ---                      ┆ ---                              │
│ str         ┆ str                  ┆ str                      ┆ str                              │
╞═════════════╪══════════════════════╪══════════════════════════╪══════════════════════════════════╡
│ _1f-o0nqpEI ┆ 2025-02-03T00:12:13Z ┆ DeepSeek, China, OpenAI, ┆ - The following is a             │
│             ┆                      ┆ NVIDIA,…                 ┆ conversatio…                     │
│ OHWnPOKh_S0 ┆ 2025-01-26T20:47:42Z ┆ Marc Andreessen: Trump,  ┆ I mean look we're adding a       │
│             ┆                      ┆ Power, T…                ┆ trill…                           │
│ Rz-4ulRKnz4 ┆ 2025-01-19T19:27:52Z ┆ Jennifer Burns: Milton   ┆ - The follo

In [8]:
# write data to file
df.write_parquet('video-transcripts.parquet')
df.write_csv('video-transcripts.csv')